# LED Matrix Client Demo

Demonstrates the `client` library. Run from the repo root or add it to your path.

**Requirements:** `pip install bleak`

In [17]:
import sys
sys.path.insert(0, "..")

import asyncio
from client import (
    LEDMatrixClient,
    bitmap_to_frame,
    frame_to_bitmap,
    print_frame,
    preview_text,
    scroll_text,
    animations,
)

## Connect

In [18]:
matrix = await LEDMatrixClient.connect()

Scanning for 'LEDMatrix'...
Found: LEDMatrix (86B8EF85-D...)
Connected: True


## Status

In [19]:
print(await matrix.read_status())

{'connected': True, 'frame_count': 0, 'last_error': 0}


## Send Frames

In [20]:
# All on
await matrix.send_frame([0xFFFFFFFF] * 8)

In [21]:
# All off
await matrix.send_frame([0x00000000] * 8)

In [22]:
# Checkerboard
checkerboard = [0xAAAAAAAA if r % 2 == 0 else 0x55555555 for r in range(8)]
print_frame(checkerboard)
await matrix.send_frame(checkerboard)

.#.#.#.#.#.#.#.#.#.#.#.#.#.#.#.#
#.#.#.#.#.#.#.#.#.#.#.#.#.#.#.#.
.#.#.#.#.#.#.#.#.#.#.#.#.#.#.#.#
#.#.#.#.#.#.#.#.#.#.#.#.#.#.#.#.
.#.#.#.#.#.#.#.#.#.#.#.#.#.#.#.#
#.#.#.#.#.#.#.#.#.#.#.#.#.#.#.#.
.#.#.#.#.#.#.#.#.#.#.#.#.#.#.#.#
#.#.#.#.#.#.#.#.#.#.#.#.#.#.#.#.


## Draw with a Bitmap

Edit the 8×32 array of 0s and 1s and run the send cell.

In [7]:
import numpy as np

canvas = np.zeros((8, 32), dtype=int)

# Draw a border
canvas[0, :] = 1
canvas[7, :] = 1
canvas[:, 0] = 1
canvas[:, 31] = 1

print("Preview:")
for row in canvas:
    print("".join("#" if p else "." for p in row))

Preview:
################################
#..............................#
#..............................#
#..............................#
#..............................#
#..............................#
#..............................#
################################


In [8]:
await matrix.send_frame(bitmap_to_frame(canvas.tolist()))

## Text

In [23]:
preview_text("HELLO WORLD")

Rendered 'HELLO WORLD' → 98 columns

.##..##...######...##.......##........####.............##...##...####....#####....##.......####...
.##..##...##.......##.......##.......##..##............##...##..##..##...##..##...##.......##.##..
.##..##...##.......##.......##.......##..##............##...##..##..##...##..##...##.......##..##.
.######...####.....##.......##.......##..##............##.#.##..##..##...#####....##.......##..##.
.##..##...##.......##.......##.......##..##............#######..##..##...####.....##.......##..##.
.##..##...##.......##.......##.......##..##............###.###..##..##...##.##....##.......##.##..
.##..##...######...######...######....####.............##...##...####....##..##...######...####...
..................................................................................................


In [24]:
await scroll_text(matrix, "HELLO WORLD", delay=0.05)

## Animations

In [11]:
await animations.row_scan(matrix, delay=0.05)

In [12]:
await animations.column_scan(matrix, delay=0.05)

In [13]:
await animations.blink(matrix, [0xFFFFFFFF] * 8, times=5)

## Status Notifications

In [14]:
await matrix.subscribe_status(lambda s: print("Status:", s))
# Send a frame to trigger an update
await matrix.send_frame([0xFFFFFFFF] * 8)

Status: {'connected': True, 'frame_count': 186, 'last_error': 0}


In [15]:
await matrix.unsubscribe_status()

## Disconnect

In [25]:
await matrix.disconnect()

Disconnected: True
